In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms, models
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from collections import Counter
import optuna
import shutil
import os
import time
from tqdm.auto import tqdm
import torch.nn.functional as F

In [2]:
train_dir ='train'

Data Collection

In [3]:
normalize_stats =((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

In [10]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Normalize(*normalize_stats)
])


In [4]:
dataset =datasets.ImageFolder(train_dir, transform=train_transforms)
train_loader =DataLoader(dataset, batch_size =32, shuffle=True)
print(f"พบรูปทั้งหมด {len(dataset)} รูป")
print(f"มี Class ทั้งหมด {len(dataset.classes)}")

NameError: name 'train_transforms' is not defined

In [12]:
targets = dataset.targets 
class_counts = Counter(targets)

df = pd.DataFrame({
    "Class Name": dataset.classes,
    "Count": [class_counts[i] for i in range(len(dataset.classes))]
})

df = df.sort_values(by="Count", ascending=False)
print(df)

  Class Name  Count
0       acne    300
2     herpes    300
4    rosacea    300
1      eksim    297
3       panu    297


In [13]:
device =torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"กำลังใช้ {device} ประมวลผล")

กำลังใช้ cuda ประมวลผล


In [14]:
class VitModel(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.3):
        super().__init__()
        self.backbone = models.vit_b_16(weights='DEFAULT')

        for param in self.backbone.parameters():
            param.requires_grad = False

        input_dim = self.backbone.heads[0].in_features

        self.backbone.heads = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

In [15]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0, verbose=False, path='checkpoint.pth'):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_acc_max = -np.inf

    def __call__(self, val_acc, model, path=None):
        if path: self.path = path
        
        score = val_acc

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
        elif score < self.best_score + self.delta: 
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_acc, model)
            self.counter = 0

    def save_checkpoint(self, val_acc, model):
        torch.save(model.state_dict(), self.path)
        self.val_acc_max = val_acc

In [ ]:
def objective(trial):
    # ==========================
    # 🎯 1. Optuna Hyperparameters
    # ==========================
    lr = trial.suggest_float("lr", 5e-6, 5e-5, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 0.1)

    # Gradient Accumulation
    accum_steps = 4 if batch_size == 16 else 2

    print(f"\n🚀 Trial {trial.number}: LR={lr:.6f}, BS={batch_size}, Drop={dropout:.2f}")

    # ==========================
    # 🔄 2. K-Fold Setup
    # ==========================
    N_FOLDS = 3
    N_EPOCHS = 15

    all_labels = dataset.targets
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_accuracies = []

    # Fold Loop
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(all_labels)), all_labels)):
        print(f"  📂 Fold {fold+1}/{N_FOLDS}")

        # Prepare Data
        train_sub = Subset(dataset, train_idx)
        val_sub = Subset(dataset, val_idx)

        train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False, num_workers=2)

        model = VitModel(num_classes=5, dropout_rate=dropout).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()
        scaler = GradScaler()
        early_stopping = EarlyStopping(patience=2, delta=0.01)

        # --- Training Loop ---
        for epoch in range(N_EPOCHS):
            model.train()
            optimizer.zero_grad()

            for step, (images, labels) in enumerate(train_loader):
                images, labels = images.to(device), labels.to(device)

                # ⚡ Mixed Precision
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    loss = loss / accum_steps

                scaler.scale(loss).backward()

                if (step + 1) % accum_steps == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            # --- Validation Loop ---
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)

                    outputs = model(images)
                    _, predicted = torch.max(outputs, 1)

                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            acc = correct / total

            save_path = f"model_trial{trial.number}_fold{fold}.pth"
            early_stopping(acc, model, save_path)

            if early_stopping.early_stop:
                print(f"    🛑 Early stopping at epoch {epoch+1}")
                break

        fold_accuracies.append(early_stopping.best_score)

        # Pruning
        trial.report(early_stopping.best_score, fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_accuracies)

In [17]:
study = optuna.create_study(direction="maximize")

print("🌪️ Starting Ultimate Training Pipeline...")
study.optimize(objective, n_trials=5)

print("\n🏆 BEST RESULT 🏆")
print(f"Best Accuracy: {study.best_value:.4f}")
print("Best Hyperparameters:", study.best_params)

[I 2026-02-16 16:03:02,789] A new study created in memory with name: no-name-75ad8b6a-1b84-45bd-9f1f-6a1c80e039d7


🌪️ Starting Ultimate Training Pipeline...

🚀 Trial 0: LR=0.000009, BS=32, Drop=0.06
  📂 Fold 1/3


C:\Users\Meepa\AppData\Local\Temp\ipykernel_28528\2221611953.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
C:\Users\Meepa\AppData\Local\Temp\ipykernel_28528\2221611953.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  📂 Fold 2/3
  📂 Fold 3/3


[I 2026-02-16 16:09:35,312] Trial 0 finished with value: 0.5870147255689425 and parameters: {'lr': 9.495732218275998e-06, 'batch_size': 32, 'dropout': 0.0567694907361754, 'weight_decay': 0.0704706524599721}. Best is trial 0 with value: 0.5870147255689425.


...


🚀 Trial 4: LR=0.000006, BS=16, Drop=0.03
  📂 Fold 1/3


C:\Users\Meepa\AppData\Local\Temp\ipykernel_28528\2221611953.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
C:\Users\Meepa\AppData\Local\Temp\ipykernel_28528\2221611953.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  📂 Fold 2/3
  📂 Fold 3/3


[I 2026-02-16 16:35:04,362] Trial 4 finished with value: 0.5026773761713521 and parameters: {'lr': 5.942906950203671e-06, 'batch_size': 16, 'dropout': 0.03250510403689002, 'weight_decay': 0.017742954432734496}. Best is trial 3 with value: 0.7637215528781794.



🏆 BEST RESULT 🏆
Best Accuracy: 0.7637
Best Hyperparameters: {'lr': 4.062316586810625e-05, 'batch_size': 32, 'dropout': 0.22942990743114866, 'weight_decay': 0.05371858111557863}
